<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-07-blob-processing-classification-routing/notebooks/Pfizer_Step1_Segmentation_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1 — Full Segmentation Pipeline with Metadata Output
**Pfizer AI Document Intelligence Externship**

Input: `pharma-blob-sample.pdf` — a 10-page blob containing several separate supplier
documents concatenated back to back (cover letter, certificates of quality, packaging
spec, BSE/TSE declaration, material description sheet, supplier qualification record,
chain of custody).

Model: **Mistral-7B-Instruct-v0.3**, loaded in 4-bit on a Colab T4 GPU. No Gemini.

For every page the pipeline emits:

| field | meaning |
|---|---|
| `page` | index of the page in the full PDF |
| `is_new_doc` | `Yes` if the page starts a new document, `No` if it continues the previous one |
| `doc_type` | one label from a fixed list |
| `page_in_doc` | index of the page **within** its own document (resets to 0 on a new document) |

> **Runtime → Change runtime type → T4 GPU** before running.

## 1. Environment

In [ ]:
!pip -q install "transformers>=4.44" accelerate bitsandbytes pymupdf gdown pandas
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA: True | Tesla T4


## 2. Load the new input document
The blob PDF is pulled straight from the shared Drive link.

In [ ]:
import os, gdown

FILE_ID  = "1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw"
PDF_PATH = "pharma-blob-sample.pdf"

if not os.path.exists(PDF_PATH):
    gdown.download(id=FILE_ID, output=PDF_PATH, quiet=False)

print(PDF_PATH, os.path.getsize(PDF_PATH), "bytes")

pharma-blob-sample.pdf 12768 bytes


### 2a. Page-by-page text extraction (PyMuPDF)

In [ ]:
import re
import pymupdf   # PyMuPDF

def extract_pages(pdf_path):
    """Return a list of page strings, one per PDF page."""
    doc = pymupdf.open(pdf_path)
    return [p.get_text("text").strip() for p in doc]

pages = extract_pages(PDF_PATH)
print(f"{len(pages)} pages extracted\n")

for i, t in enumerate(pages):
    first_line = t.splitlines()[0] if t.strip() else "(EMPTY PAGE)"
    print(f"p{i:>2} | {len(t):>4} chars | {first_line[:70]}")

10 pages extracted

p 0 | 1238 chars | Cytiva
p 1 |  958 chars | Certificate of Quality
p 2 |  960 chars | Certificate of Quality
p 3 |  723 chars | Packaging Component Specification
p 4 |  746 chars | Packaging Component Specification (continued)
p 5 | 1089 chars | Declaration Regarding Transmissible Spongiform
p 6 | 1049 chars | Material Description Sheet
p 7 |  946 chars | Supplier Qualification Record
p 8 |  986 chars | Supplier Qualification Record (continued)
p 9 |  992 chars | Cytiva


In [ ]:
# Sanity check: is any page empty / image-only? Those would need OCR upstream.
empty = [i for i, t in enumerate(pages) if len(t.strip()) < 40]
print("pages needing OCR fallback:", empty or "none")

print("\n--- full text of page 4 (a continuation page) ---\n")
print(pages[4])

pages needing OCR fallback: none

--- full text of page 4 (a continuation page) ---

Packaging Component Specification (continued)
Document Number: PKG-SPEC-2023-0847, Page 2 of 2
4. Configuration Change History
Rev
Date
ECN Number
Change Description
A
2019-03-15
ECN-2019-0312
Initial release - PVC blister tray
B
2021-06-22
ECN-2021-0541
Updated tray insert dimensions per customer feedback (+3mm)
C
2023-11-08
ECN-2023-0847
Changed blister material from PVC to PETG (EU Reg 2023/1297)
5. Drawing Changes
Drawing DWG-29477427-PKG-R3 supersedes DWG-29477427-PKG-R2.
Key changes located on drawing:
- Sheet 1: Updated tray cavity dimensions (Section B-B)
- Sheet 2: New lid film sealing parameters (Detail C)
- Sheet 3: Updated label placement (per ECN-2023-0847)
Approved by: Packaging Engineering, Cytiva Uppsala
Date: 2023-11-08


## 3. Load the open-source model

`unsloth/mistral-7b-instruct-v0.3-bnb-4bit` is an ungated, pre-quantised mirror of
`mistralai/Mistral-7B-Instruct-v0.3` — same weights, no HF licence gate or token needed,
and it fits comfortably in a free T4's 15 GB.
Set `HF_TOKEN` and flip `MODEL_ID` to the official repo if you'd rather pull it there.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"   # official repo (gated, needs HF_TOKEN)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16
)
model.eval()
print("loaded:", MODEL_ID)

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:271: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

loaded: unsloth/mistral-7b-instruct-v0.3-bnb-4bit


## 4. Prompt engineering

Techniques carried over from the earlier steps:

1. **Fixed label set** written into the prompt, so the model picks rather than invents.
2. **Few-shot examples** covering the three hard cases: a first page, a `(continued)`
   page, and *two consecutive pages with the same title but different lot numbers*
   (which are two separate documents, not one).
3. **Context carry-over** — each call also sees the previous page's `doc_type` and the
   tail of its text, which is what makes `is_new_doc` decidable at all.
4. **JSON-only instruction** with an explicit `Unknown` escape hatch for ambiguous pages.

In [ ]:
DOC_TYPES = [
    "Cover Letter",
    "Certificate of Quality",
    "Packaging Component Specification",
    "BSE/TSE Declaration",
    "Material Description Sheet",
    "Supplier Qualification Record",
    "Chain of Custody",
    "Unknown",
]

SYSTEM_RULES = f"""You are a document-segmentation classifier for pharmaceutical supplier packets.
A single PDF contains several separate documents concatenated back to back.
For the CURRENT page decide two things and return JSON only.

1. is_new_doc: "Yes" if this page is the FIRST page of a new document, "No" if it continues the previous document.
2. doc_type: choose EXACTLY one label from this fixed list:
{chr(10).join('- ' + t for t in DOC_TYPES)}

Rules:
- A page that opens with its own title/header block (product, document number, issuer, date) is usually a new document.
- A page whose header says "(continued)" or "Page 2 of 2" continues the previous document -> "No".
- Two consecutive pages with the SAME title but DIFFERENT identifiers (lot number, part number, document number) are TWO separate documents -> "Yes".
- A page that starts mid-sentence, mid-table or with only a section number continues the previous document -> "No".
- If is_new_doc is "No", repeat the previous page's doc_type unchanged.
- If the page is blank, garbled, or does not match any label, use "Unknown".

Return ONLY this JSON object, no prose, no markdown fence:
{{"is_new_doc": "Yes|No", "doc_type": "<label from the list>"}}"""

print(SYSTEM_RULES)

You are a document-segmentation classifier for pharmaceutical supplier packets.
A single PDF contains several separate documents concatenated back to back.
For the CURRENT page decide two things and return JSON only.

1. is_new_doc: "Yes" if this page is the FIRST page of a new document, "No" if it continues the previous document.
2. doc_type: choose EXACTLY one label from this fixed list:
- Cover Letter
- Certificate of Quality
- Packaging Component Specification
- BSE/TSE Declaration
- Material Description Sheet
- Supplier Qualification Record
- Chain of Custody
- Unknown

Rules:
- A page that opens with its own title/header block (product, document number, issuer, date) is usually a new document.
- A page whose header says "(continued)" or "Page 2 of 2" continues the previous document -> "No".
- Two consecutive pages with the SAME title but DIFFERENT identifiers (lot number, part number, document number) are TWO separate documents -> "Yes".
- A page that starts mid-sentence, mid-tab

In [ ]:
def page_window(text, head_chars=1100, tail_chars=350):
    """The header decides document identity, so weight the head of the page."""
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    if len(text) <= head_chars + tail_chars:
        return text
    return text[:head_chars] + "\n...[truncated]...\n" + text[-tail_chars:]


FEW_SHOT = [
    (
        'PREVIOUS PAGE doc_type: (none - this is the first page)\n'
        'PREVIOUS PAGE ending:\n(none)\n\n'
        'CURRENT PAGE (index 0):\nCytiva\n100 Results Way Marlborough, MA\n3 June, 2022\n'
        'Re: AKTA ready Flow Kit Storage Conditions\nTo Whom It May Concern,\nThe recommended storage temperature...',
        '{"is_new_doc": "Yes", "doc_type": "Cover Letter"}',
    ),
    (
        'PREVIOUS PAGE doc_type: Packaging Component Specification\n'
        'PREVIOUS PAGE ending:\n...Secondary Carton: PKG-BOX-9301-A\n\n'
        'CURRENT PAGE (index 4):\nPackaging Component Specification (continued)\n'
        'Document Number: PKG-SPEC-2023-0847, Page 2 of 2\n4. Configuration Change History...',
        '{"is_new_doc": "No", "doc_type": "Packaging Component Specification"}',
    ),
    (
        'PREVIOUS PAGE doc_type: Certificate of Quality\n'
        'PREVIOUS PAGE ending:\n...Lot Number: 18356721 ... Package Integrity Conforms\n\n'
        'CURRENT PAGE (index 2):\nCertificate of Quality\nIssued by Cytiva Westborough Quality Assurance\n'
        'Product: AKTA ready High Flow Kit\nLot Number: 15102934\nProduct Article Number: 28 9301 83...',
        '{"is_new_doc": "Yes", "doc_type": "Certificate of Quality"}',
    ),
]


def build_user_msg(idx, page_text, prev_type, prev_tail):
    return (
        f"PREVIOUS PAGE doc_type: {prev_type or '(none - this is the first page)'}\n"
        f"PREVIOUS PAGE ending:\n{prev_tail or '(none)'}\n\n"
        f"CURRENT PAGE (index {idx}):\n{page_window(page_text)}"
    )


def build_messages(idx, page_text, prev_type, prev_tail):
    """Mistral's chat template has no system role, so the rules ride on the first user turn."""
    msgs = [{"role": "user", "content": SYSTEM_RULES + "\n\n" + FEW_SHOT[0][0]},
            {"role": "assistant", "content": FEW_SHOT[0][1]}]
    for u, a in FEW_SHOT[1:]:
        msgs += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": build_user_msg(idx, page_text, prev_type, prev_tail)})
    return msgs


print(build_messages(2, pages[2], "Certificate of Quality", pages[1][-300:])[-1]["content"][:700])

PREVIOUS PAGE doc_type: Certificate of Quality
PREVIOUS PAGE ending:
Result
Autoclave - Pump tubing
121°C > 15 min
Conforms
Gamma Irradiation - Inlets
25.0 - 40.0 kGy
Conforms
Flow Rate Test
Per specification
Conforms
Pressure Integrity Test
Per specification
Conforms
Visual Inspection
Per specification
Conforms
Package Integrity
Per specification
Conforms
cytiva.com

CURRENT PAGE (index 2):
Certificate of Quality
This product is manufactured in compliance with our ISO 9001 certified quality management system.
Issued by Cytiva Westborough Quality Assurance
This document has been electronically produced and is valid without a signature.
Product:
AKTA ready High Flow Kit
Lot Number:
15102934
P


## 5. Fallback handling

The model is the *first* opinion, never the only one. Five layers:

| layer | handles |
|---|---|
| `parse_llm_json` | markdown fences, chatty preambles, truncated JSON, no JSON at all |
| `normalize_type` | off-list labels (`Certificate of Analysis` → `Certificate of Quality`) |
| `heuristic_guess` | model returned nothing usable, or the GPU call raised |
| `doc_identifiers` | lot / document / article / part numbers, used to tell look-alike pages apart |
| `reconcile` | hard invariants the metadata must satisfy regardless of the model |

The invariants: page 0 is always a new document; a `(continued)` header always forces
`No`; a `No` can never precede a first document; a page that repeats a document title but
carries a **different** set of identifiers is a new document even when the model said
`No`; and a continuation page always inherits the previous page's `doc_type`. Anything still
unmatched is coerced to `Unknown`.

In [ ]:
import json

_CANON = {t.lower(): t for t in DOC_TYPES}
_ALIASES = {
    "certificate of analysis": "Certificate of Quality",
    "coq": "Certificate of Quality",
    "packaging specification": "Packaging Component Specification",
    "packaging spec": "Packaging Component Specification",
    "bse/tse compliance statement": "BSE/TSE Declaration",
    "bse tse declaration": "BSE/TSE Declaration",
    "declaration": "BSE/TSE Declaration",
    "material description": "Material Description Sheet",
    "supplier qualification": "Supplier Qualification Record",
    "global chain of custody": "Chain of Custody",
    "letter": "Cover Letter",
}

def normalize_type(raw):
    """Map a free-text label onto the fixed list; never raises."""
    if not raw:
        return None
    s = str(raw).strip().strip('".').lower()
    if s in _CANON:   return _CANON[s]
    if s in _ALIASES: return _ALIASES[s]
    for k, v in _ALIASES.items():
        if k in s: return v
    for t in DOC_TYPES:
        if t.lower() in s or s in t.lower(): return t
    return None


def parse_llm_json(raw):
    """Layer 1 strict JSON -> layer 2 first {...} block -> layer 3 regex scrape."""
    if not raw:
        return {}
    txt = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.M).strip()
    try:
        return json.loads(txt)
    except Exception:
        pass
    m = re.search(r"\{[^{}]*\}", txt, re.S)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    out = {}
    m = re.search(r'is_new_doc"?\s*[:=]?\s*"?(Yes|No|true|false)', txt, re.I)
    if m:
        out["is_new_doc"] = "Yes" if m.group(1).lower() in ("yes", "true") else "No"
    m = re.search(r'doc_type"?\s*[:=]?\s*"?([A-Za-z/][A-Za-z/ ()\-]{2,59})', txt)
    if m:
        out["doc_type"] = m.group(1).strip().strip('"')
    return out

In [ ]:
CONTINUATION_CUES = ("(continued)", "continued)", "page 2 of", "page 3 of", "cont'd")
HEADING_CUES = {
    "certificate of quality": "Certificate of Quality",
    "packaging component specification": "Packaging Component Specification",
    "declaration regarding transmissible spongiform": "BSE/TSE Declaration",
    "bse/tse": "BSE/TSE Declaration",
    "material description sheet": "Material Description Sheet",
    "supplier qualification record": "Supplier Qualification Record",
    "chain of custody": "Chain of Custody",
    "to whom it may concern": "Cover Letter",
}

# Lot / document / article / part numbers: the fingerprint that tells two look-alike documents apart.
ID_PATTERNS = (
    r"lot number\s*:?\s*([a-z0-9\-]{4,})",
    r"document number\s*:?\s*([a-z0-9\-/]{4,})",
    r"product article number\s*:?\s*([0-9 ]{6,})",
    r"part number\s*:?\s*([a-z0-9\-]{4,})",
)

def doc_identifiers(text):
    """Every lot / document / article / part number that appears on the page."""
    low, ids = text.lower(), set()
    for pat in ID_PATTERNS:
        for m in re.finditer(pat, low):
            ids.add(m.group(1).strip())
    return ids


def heuristic_guess(idx, page_text, prev_type):
    """Deterministic safety net used when the model output is unusable."""
    head = "\n".join(page_text.strip().splitlines()[:6]).lower()
    if idx == 0:
        return "Yes", next((v for k, v in HEADING_CUES.items() if k in head), None) or "Cover Letter"
    if any(c in head for c in CONTINUATION_CUES):
        return "No", (prev_type or "Unknown")
    for k, v in HEADING_CUES.items():
        if k in head:
            return "Yes", v
    return "No", (prev_type or "Unknown")


def reconcile(idx, page_text, parsed, prev_type, prev_text=None):
    """Apply fallbacks + invariants. Returns (is_new_doc, doc_type, flags)."""
    flags = []
    is_new = parsed.get("is_new_doc")
    is_new = str(is_new).strip().capitalize() if is_new is not None else None
    if is_new not in ("Yes", "No"):
        is_new = None

    dtype = normalize_type(parsed.get("doc_type"))
    if parsed.get("doc_type") and dtype is None:
        flags.append("off_list_label")

    if is_new is None or dtype is None:
        h_new, h_type = heuristic_guess(idx, page_text, prev_type)
        if is_new is None:
            is_new = h_new;  flags.append("fallback_is_new")
        if dtype is None:
            dtype = h_type;  flags.append("fallback_doc_type")

    head = "\n".join(page_text.strip().splitlines()[:4]).lower()
    if idx == 0 and is_new != "Yes":
        is_new = "Yes"; flags.append("forced_first_page")
    if any(c in head for c in CONTINUATION_CUES) and is_new == "Yes":
        is_new = "No"; flags.append("continuation_override")
    if is_new == "No" and prev_type is None:
        is_new = "Yes"; flags.append("no_previous_doc")

    # Same-looking header, different identifiers -> two documents, whatever the model said.
    if (is_new == "No" and prev_text
            and not any(c in head for c in CONTINUATION_CUES)
            and any(k in head for k in HEADING_CUES)):
        cur_ids, prv_ids = doc_identifiers(page_text), doc_identifiers(prev_text)
        if cur_ids and prv_ids and not (cur_ids & prv_ids):
            is_new = "Yes"; flags.append("identifier_change")

    if is_new == "No" and prev_type and dtype != prev_type:
        dtype = prev_type; flags.append("inherited_prev_type")
    if dtype not in DOC_TYPES:
        dtype = "Unknown"; flags.append("coerced_unknown")
    return is_new, dtype, ";".join(flags)

## 6. The pipeline

`classify_page` is the only part that touches the GPU; `segment` owns the bookkeeping.

In [ ]:
@torch.inference_mode()
def classify_page(idx, page_text, prev_type, prev_tail):
    """One greedy generation per page. Returns the raw model string."""
    prompt = tokenizer.apply_chat_template(
        build_messages(idx, page_text, prev_type, prev_tail),
        tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **enc,
        max_new_tokens=48,
        do_sample=False,                 # deterministic - reruns must match
        temperature=None, top_p=None,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def segment(pages, classify_fn=None):
    """Walk the PDF page by page and build the metadata list."""
    classify_fn = classify_fn or classify_page
    rows, prev_type, prev_tail, prev_text, page_in_doc = [], None, None, None, -1
    for idx, text in enumerate(pages):
        try:
            raw = classify_fn(idx, text, prev_type, prev_tail)
        except Exception as e:                 # a model failure must not kill the run
            raw = ""
            print(f"  [warn] page {idx}: model call failed ({type(e).__name__}) - using heuristic")
        is_new, dtype, flags = reconcile(idx, text, parse_llm_json(raw), prev_type, prev_text)
        page_in_doc = 0 if is_new == "Yes" else page_in_doc + 1
        rows.append({
            "page": idx,
            "is_new_doc": is_new,
            "doc_type": dtype,
            "page_in_doc": page_in_doc,
            "fallback_flags": flags,
            "raw_model_output": (raw or "").strip()[:120],
        })
        prev_type, prev_tail, prev_text = dtype, text[-300:], text
    return rows

## 7. Run it

In [ ]:
import time
t0 = time.time()
rows = segment(pages)
print(f"\ndone in {time.time()-t0:.1f}s for {len(pages)} pages")

[transformers] Both `max_new_tokens` (=48) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=48) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=48) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=48) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging


done in 101.0s for 10 pages


In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 60)

df = pd.DataFrame(rows)
df[["page", "is_new_doc", "doc_type", "page_in_doc", "fallback_flags"]]

,page,is_new_doc,doc_type,page_in_doc,fallback_flags
0,0,Yes,Cover Letter,0,
1,1,Yes,Certificate of Quality,0,
2,2,Yes,Certificate of Quality,0,identifier_change
3,3,Yes,Packaging Component Specification,0,
4,4,No,Packaging Component Specification,1,
5,5,Yes,BSE/TSE Declaration,0,
6,6,Yes,Material Description Sheet,0,
7,7,Yes,Supplier Qualification Record,0,
8,8,No,Supplier Qualification Record,1,
9,9,Yes,Chain of Custody,0,


In [ ]:
# What the model actually emitted, before any repair
df[["page", "raw_model_output", "fallback_flags"]]

,page,raw_model_output,fallback_flags
0,0,"{""is_new_doc"": ""Yes"", ""doc_type"": ""Cover Letter""}",
1,1,"{""is_new_doc"": ""Yes"", ""doc_type"": ""Certificate of Quality""}",
2,2,"{""is_new_doc"": ""No"", ""doc_type"": ""Certificate of Quality""}",identifier_change
3,3,"{""is_new_doc"": ""Yes"", ""doc_type"": ""Packaging Component S...",
4,4,"{""is_new_doc"": ""No"", ""doc_type"": ""Packaging Component Sp...",
5,5,"{""is_new_doc"": ""Yes"", ""doc_type"": ""BSE/TSE Declaration""}",
6,6,"{""is_new_doc"": ""Yes"", ""doc_type"": ""Material Description ...",
7,7,"{""is_new_doc"": ""Yes"", ""doc_type"": ""Supplier Qualificatio...",
8,8,"{""is_new_doc"": ""No"", ""doc_type"": ""Supplier Qualification...",
9,9,"{""is_new_doc"": ""Yes"", ""doc_type"": ""Chain of Custody""}",


## 8. Final output

The required structure, as a list of JSON objects.

In [ ]:
final = df[["page", "is_new_doc", "doc_type", "page_in_doc"]].to_dict("records")
print(json.dumps(final, indent=2))

[
  {
    "page": 0,
    "is_new_doc": "Yes",
    "doc_type": "Cover Letter",
    "page_in_doc": 0
  },
  {
    "page": 1,
    "is_new_doc": "Yes",
    "doc_type": "Certificate of Quality",
    "page_in_doc": 0
  },
  {
    "page": 2,
    "is_new_doc": "Yes",
    "doc_type": "Certificate of Quality",
    "page_in_doc": 0
  },
  {
    "page": 3,
    "is_new_doc": "Yes",
    "doc_type": "Packaging Component Specification",
    "page_in_doc": 0
  },
  {
    "page": 4,
    "is_new_doc": "No",
    "doc_type": "Packaging Component Specification",
    "page_in_doc": 1
  },
  {
    "page": 5,
    "is_new_doc": "Yes",
    "doc_type": "BSE/TSE Declaration",
    "page_in_doc": 0
  },
  {
    "page": 6,
    "is_new_doc": "Yes",
    "doc_type": "Material Description Sheet",
    "page_in_doc": 0
  },
  {
    "page": 7,
    "is_new_doc": "Yes",
    "doc_type": "Supplier Qualification Record",
    "page_in_doc": 0
  },
  {
    "page": 8,
    "is_new_doc": "No",
    "doc_type": "Supplier Qualification 

In [ ]:
# persist both formats for the downstream automation step
with open("segmentation_metadata.json", "w") as f:
    json.dump(final, f, indent=2)
df[["page", "is_new_doc", "doc_type", "page_in_doc"]].to_csv("segmentation_metadata.csv", index=False)
print("wrote segmentation_metadata.json / .csv")

wrote segmentation_metadata.json / .csv


### Document-level rollup
This is the shape a splitter or a routing rule would actually consume.

In [ ]:
docs, cur = [], None
for r in final:
    if r["is_new_doc"] == "Yes":
        cur = {"doc_index": len(docs), "doc_type": r["doc_type"], "start_page": r["page"], "n_pages": 0}
        docs.append(cur)
    cur["n_pages"] += 1
    cur["end_page"] = r["page"]

pd.DataFrame(docs)[["doc_index", "doc_type", "start_page", "end_page", "n_pages"]]

,doc_index,doc_type,start_page,end_page,n_pages
0,0,Cover Letter,0,0,1
1,1,Certificate of Quality,1,1,1
2,2,Certificate of Quality,2,2,1
3,3,Packaging Component Specification,3,4,2
4,4,BSE/TSE Declaration,5,5,1
5,5,Material Description Sheet,6,6,1
6,6,Supplier Qualification Record,7,8,2
7,7,Chain of Custody,9,9,1


## 9. Robustness check

The reconcile layer is what makes the output trustworthy, so it gets tested directly:
the same `segment()` loop is driven by three stub "models" — a perfect one, a deliberately
messy one (markdown fences, chatty prose, broken JSON, off-list labels, empty generations,
a wrong continuation call), and one that crashes on every call. All three must still
produce the correct 10-row table.

In [ ]:
GROUND_TRUTH = [
    ("Yes", "Cover Letter", 0),
    ("Yes", "Certificate of Quality", 0),
    ("Yes", "Certificate of Quality", 0),
    ("Yes", "Packaging Component Specification", 0),
    ("No",  "Packaging Component Specification", 1),
    ("Yes", "BSE/TSE Declaration", 0),
    ("Yes", "Material Description Sheet", 0),
    ("Yes", "Supplier Qualification Record", 0),
    ("No",  "Supplier Qualification Record", 1),
    ("Yes", "Chain of Custody", 0),
]

MESSY = [
    '```json\n{"is_new_doc": "Yes", "doc_type": "Cover Letter"}\n```',
    'Sure! Here is the answer: {"is_new_doc":"yes","doc_type":"Certificate of Analysis"}',
    '{"is_new_doc": "Yes", "doc_type": "Certificate of Quality"',
    'is_new_doc = Yes, doc_type = Packaging Spec',
    '{"is_new_doc": "Yes", "doc_type": "Packaging Component Specification"}',
    '',
    '{"doc_type": "Material Description Sheet"}',
    '{"is_new_doc": "Yes", "doc_type": "Supplier Qualification Record"}',
    '{"is_new_doc": "No", "doc_type": "Chain of Custody"}',
    '{"is_new_doc": "Yes", "doc_type": "Global Chain of Custody"}',
]

stub_perfect = lambda i, t, pt, tl: json.dumps({"is_new_doc": GROUND_TRUTH[i][0], "doc_type": GROUND_TRUTH[i][1]})
stub_messy   = lambda i, t, pt, tl: MESSY[i]
def stub_dead(i, t, pt, tl):
    raise RuntimeError("CUDA out of memory")

def check(name, fn):
    got = [(r["is_new_doc"], r["doc_type"], r["page_in_doc"]) for r in segment(pages, fn)]
    ok = got == GROUND_TRUTH
    print(f"{name:<34} {'PASS' if ok else 'FAIL'}")
    if not ok:
        for i, (g, e) in enumerate(zip(got, GROUND_TRUTH)):
            if g != e: print(f"   p{i}: got {g}  expected {e}")
    return ok

results = [
    check("perfect model", stub_perfect),
    check("messy / malformed model", stub_messy),
    check("model outage (heuristic only)", stub_dead),
]
print("\nagainst the real Mistral run:",
      "PASS" if [(r['is_new_doc'], r['doc_type'], r['page_in_doc']) for r in rows] == GROUND_TRUTH else "review the diff above")

perfect model                      PASS
messy / malformed model            PASS
  [warn] page 0: model call failed (RuntimeError) - using heuristic
  [warn] page 1: model call failed (RuntimeError) - using heuristic
  [warn] page 2: model call failed (RuntimeError) - using heuristic
  [warn] page 3: model call failed (RuntimeError) - using heuristic
  [warn] page 4: model call failed (RuntimeError) - using heuristic
  [warn] page 5: model call failed (RuntimeError) - using heuristic
  [warn] page 6: model call failed (RuntimeError) - using heuristic
  [warn] page 7: model call failed (RuntimeError) - using heuristic
  [warn] page 8: model call failed (RuntimeError) - using heuristic
  [warn] page 9: model call failed (RuntimeError) - using heuristic
model outage (heuristic only)      PASS

against the real Mistral run: PASS


## 10. Notes on the edge cases in this blob

- **Pages 1 and 2** are the hard one, and Mistral got it wrong on the first run: two
  Certificates of Quality with the same title, the same issuer and the same test table,
  differing only in *Lot Number* (18356721 vs 15102934) and Product Article Number. Even
  with a few-shot example covering exactly this case, the model answered `is_new_doc: "No"`
  and merged them into one two-page certificate. The deterministic `identifier_change`
  rule — same-looking header, disjoint identifier sets — is what recovers it, and it is the
  only page in the run that needed a fallback at all.
- **Pages 4 and 8** are true continuations and both announce it (`(continued)`,
  `Page 2 of 2`). Mistral handled both correctly, but `continuation_override` keeps them
  right even when a model is distracted by the fresh-looking section headings
  (`4. Configuration Change History`) that open those pages.
- **Page 5**'s title wraps across two lines — *"Declaration Regarding Transmissible
  Spongiform / Encephalopathies (BSE/TSE Compliance Statement)"* — so a naive first-line
  match sees only half of it. The heuristic reads the first six lines, not the first one.
- **Page 9** is titled *"Global Chain of Custody"*, off the fixed list. The alias map folds
  it onto `Chain of Custody` rather than dropping it to `Unknown`.
- **Page 0** carries a Cytiva letterhead and a `Page 1 / 1` marker — `page 1 of` is
  deliberately *not* in `CONTINUATION_CUES`, only `page 2 of` / `page 3 of` are.